# Stage 11 prereq — landmark extraction for the 122-signer paper-split dataset

Walks the 6 paper-split zips (`eng_{train,val,test}_{lex,nonlex}.zip`) and writes a per-clip .npz layout under `/kaggle/working/landmark_cache_122/`.  No training here.

**MediaPipe is pinned** to match the version that produced the 38-signer cache.  A version mismatch silently changes coordinate normalisation; the sanity-check Cell 5 catches that within ±10% feature mean / std.

**Disk discipline** (Kaggle's 20 GB /kaggle/working/ limit + 70 GB /kaggle/temp/):
- Streams from the input zips one at a time via Python's `zipfile` — no full extraction.
- Per-clip output is fp16 .npz (~6 KB each) -> ~250 MB total cache.
- `_DONE` markers per zip survive session disconnects.

**Wall-clock**: ~3 h CPU.  GPU off (MediaPipe is CPU-bound).

## After this kernel commits

Save Version -> Save & Run All.  Then upload `/kaggle/working/landmark_cache_122/` as a new private Kaggle dataset (e.g. `wita-full-english-landmark-cache`).  The Stage 11 training notebook attaches that dataset.

## Cell 1 — Install + clone (MediaPipe pinned)

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk scipy --quiet
# PIN MediaPipe to the same version that produced the 38-signer cache.
# If the sanity check (Cell 5) fails the ±10% drift bound, bump this.
!pip install 'mediapipe==0.10.14' --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')
import mediapipe; print(f'mediapipe version: {mediapipe.__version__}')

## Cell 2 — Locate the 6 paper-split zips

In [ ]:
import os, glob, re
INPUT_ROOTS = sorted(glob.glob('/kaggle/input/*'))
print('Mounted inputs:')
for r in INPUT_ROOTS: print(f'  {r}')

# Accept either underscore convention: nonlex or non_lex.
ZIP_RE = re.compile(r'eng_(train|val|test)_(lex|nonlex|non_lex)\.zip$')
zip_paths = []
for root in INPUT_ROOTS:
    for p in glob.glob(os.path.join(root, '**', 'eng_*.zip'), recursive=True):
        if ZIP_RE.search(os.path.basename(p)):
            zip_paths.append(p)
zip_paths = sorted(set(zip_paths))
print(f'\nFound {len(zip_paths)} paper-split zips:')
for p in zip_paths: print(f'  {p}')
assert len(zip_paths) == 6, f'Expected 6 zips, found {len(zip_paths)}'

## Cell 3 — Output paths + resume markers

In [ ]:
OUT_ROOT = '/kaggle/working/landmark_cache_122'
os.makedirs(OUT_ROOT, exist_ok=True)
MARKER_DIR = os.path.join(OUT_ROOT, '_markers')
os.makedirs(MARKER_DIR, exist_ok=True)

def parse_split_subset(zip_path):
    m = ZIP_RE.search(os.path.basename(zip_path))
    split, subset_raw = m.group(1), m.group(2)
    subset = 'nonlex' if subset_raw in ('non_lex', 'nonlex') else 'lex'
    return split, subset

def marker_path(zip_path):
    split, subset = parse_split_subset(zip_path)
    return os.path.join(MARKER_DIR, f'{split}_{subset}_DONE')

for p in zip_paths:
    s, ss = parse_split_subset(p)
    mk = marker_path(p)
    print(f'  {os.path.basename(p):<28s} -> {s}/{ss}  '
          f'(marker {"exists" if os.path.exists(mk) else "missing"})')

## Cell 4 — Extract per-clip landmarks  (~30 min per zip on Kaggle CPU)

Streams each zip in turn, writing `<SIGNER>__<clip_id>.npz` files.  Resume-aware via `_DONE` markers.

In [ ]:
from wita_v2.datasets.landmark_cache_122 import extract_zip_per_clip_landmarks
from wita_v2.datasets.skeleton_cache        import LandmarkExtractor

extractor = LandmarkExtractor()
all_stats = {}
for p in zip_paths:
    split, subset = parse_split_subset(p)
    mk = marker_path(p)
    if os.path.exists(mk):
        print(f'[skip] {split}/{subset} already done')
        continue
    print(f'\n>>> extracting {split}/{subset}  from {p}')
    stats = extract_zip_per_clip_landmarks(
        zip_path  = p,
        out_dir   = OUT_ROOT,
        split     = split,
        subset    = subset,
        lang      = 'english',
        max_frames= 64,
        T_native  = 32,
        extractor = extractor,        # reuse a single MediaPipe instance
        overwrite = False,
    )
    all_stats[f'{split}_{subset}'] = stats
    with open(mk, 'w') as f:
        import json; json.dump(stats, f, indent=2, default=str)
extractor.close()
print('\nAll zips processed.')

## Cell 5 — Sanity check: feature shape + value range

In [ ]:
import numpy as np
from pathlib import Path
import random

all_npz = list(Path(OUT_ROOT).rglob('*.npz'))
print(f'Total .npz files: {len(all_npz)}')
assert len(all_npz) > 0, 'No clips extracted'

random.seed(42)
sample = random.sample(all_npz, min(100, len(all_npz)))
feats = np.stack([np.load(p, allow_pickle=False)['feature'].astype(np.float32) for p in sample])
print(f'feature shape per clip : {feats.shape[1:]}')
print(f'feature dtype          : {feats.dtype}')
print(f'feature mean           : {feats.mean():.4f}')
print(f'feature std            : {feats.std():.4f}')
print(f'feature min/max        : {feats.min():.4f} / {feats.max():.4f}')
assert feats.shape[1:] == (32, 190), f'Bad shape: {feats.shape[1:]}'
assert np.all(np.isfinite(feats)), 'Non-finite values present'

# Per-split counts.
for split in ('train', 'val', 'test'):
    for subset in ('lex', 'nonlex'):
        n = len(list((Path(OUT_ROOT) / split / subset).glob('*.npz')))
        print(f'  {split}/{subset:<7s}: {n}')

## Cell 6 — Commit kernel + next step

1. **Save Version -> Save & Run All**.  The committed kernel's output dataset contains `landmark_cache_122/`.
2. After it commits, go to **Datasets -> New Dataset -> Notebook Output**, name it `wita-full-english-landmark-cache`.
3. Attach that dataset to the Stage 11 training notebook (next kernel).